<a href="https://colab.research.google.com/github/NanaEngo/Quantum_Agrivoltaic_PT-HOPS/blob/main/Redac_Paper1/quantum_simulations_framework/quantum_agrivoltaic_colab_launcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quantum-Enhanced Agrivoltaics — Colab Simulation Launcher

**Manuscript:** Quantum-Coherent Spectral Engineering (jz-2026-00994t)  
**Solver:** PT-HOPS/SBD via MesoHOPS  

### Usage
- **Google Colab VS Code Extension**: open this notebook in VS Code with the Colab extension active. The runtime is remote; cells run on Colab's GPU/CPU.
- **Standalone Colab**: upload to [colab.research.google.com](https://colab.research.google.com) and run all cells.
- **Local (MesoHOP-sim env)**: `mamba run -n MesoHOP-sim jupyter notebook`

> ⚠️ **OOM note**: 100 trajectories at L=10 requires ~10 GB RAM per trajectory peak. On Colab (~12 GB), use `N_TRAJ = 5` for a quick test. For publication-grade results run on local hardware or HPC.

In [17]:
import sys, os, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_NAME = "Quantum_Agrivoltaic_PT-HOPS"

if IN_COLAB:
    print("🌐 Google Colab detected — setting up environment...")

    # Check if we are already in the expected repository root directory
    if os.path.basename(os.getcwd()) != REPO_NAME:
        # If not, try to change into it, assuming Colab cloned it to /content/REPO_NAME
        expected_colab_repo_root = os.path.join("/content", REPO_NAME)
        if os.path.isdir(expected_colab_repo_root):
            os.chdir(expected_colab_repo_root)
            print(f"✅ Changed directory to repository root: {os.getcwd()}")
        else:
            # If the repo isn't where expected, then we need to clone.
            print(f"Repository '{REPO_NAME}' not found at {expected_colab_repo_root}. Attempting to clone...")
            token = None
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
                if not token:
                    print("⚠️  GITHUB_TOKEN not found in Colab secrets. Checking environment variables...")
                    token = os.environ.get("GITHUB_TOKEN")
            except Exception as e:
                print(f"⚠️  Failed to retrieve GITHUB_TOKEN from Colab secrets: {e}")
                print("Checking environment variables for GITHUB_TOKEN...")
                token = os.environ.get("GITHUB_TOKEN")

            if not token:
                print("❌ GITHUB_TOKEN is required for cloning the private repository.")
                print("Please set it in Colab Secrets (if running interactively) or as an environment variable.")
                raise RuntimeError("GitHub Token not available. Cannot clone private repository.")

            repo_url = f"https://{token}@github.com/NanaEngo/{REPO_NAME}.git"
            try:
                subprocess.check_call(["git", "clone", "--depth=1", repo_url, REPO_NAME])
                print("✅ Repository cloned successfully.")
                os.chdir(REPO_NAME) # Change to the newly cloned directory
            except subprocess.CalledProcessError as e:
                print(f"❌ Failed to clone repository: {e}")
                print("This usually means the GITHUB_TOKEN is invalid, expired, or lacks read permissions for the private repository.")
                print("Please verify your GITHUB_TOKEN.")
                raise
    else:
        print(f"✅ Already in repository root: {os.getcwd()}")

    print("📦 Installing dependencies...")
    pkgs = [
        "numpy>=2.0,<2.4", "scipy>=1.10", "pandas>=2.0", "matplotlib>=3.7",
        "pyyaml>=6.0", "numba>=0.59", "h5py>=3.7", "tqdm>=4.65",
        "psutil>=5.9", "scienceplots>=2.0", "qutip>=5.2",
    ]
    for pkg in pkgs:
        print(f"Installing {pkg}...")
        try:
            # Use subprocess.run to capture output and check return code manually
            # Set capture_output=True and text=True to get stdout/stderr as strings
            result = subprocess.run([sys.executable, "-m", "pip", "install", pkg], capture_output=True, text=True, check=False)
            if result.returncode != 0:
                print(f"❌ Failed to install {pkg}:")
                print("--- PIP STDOUT ---")
                print(result.stdout)
                print("--- PIP STDERR ---")
                print(result.stderr)
                raise subprocess.CalledProcessError(result.returncode, result.args, output=result.stdout, stderr=result.stderr)
            else:
                print(f"✅ Successfully installed {pkg}")
        except subprocess.CalledProcessError as e:
            print(f"❌ Failed to install {pkg}: {e}")
            raise # Re-raise the exception after printing the problematic package

    print("✅ Setup complete.")
else:
    print("💻 Local environment — ensure MesoHOP-sim is active.")

# Navigate to framework directory
_fw = os.path.join(os.getcwd(), "Redac_Paper1", "quantum_simulations_framework")
if os.path.isdir(_fw):
    os.chdir(_fw)
elif "quantum_simulations_framework" not in os.getcwd():
    raise RuntimeError(f"Cannot locate quantum_simulations_framework/ from {os.getcwd()}")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"📁 Working directory: {os.getcwd()}")

🌐 Google Colab detected — setting up environment...
✅ Already in repository root: /content/Quantum_Agrivoltaic_PT-HOPS
📦 Installing dependencies...
Installing numpy>=2.0,<2.4...
✅ Successfully installed numpy>=2.0,<2.4
Installing scipy>=1.10...
✅ Successfully installed scipy>=1.10
Installing pandas>=2.0...
✅ Successfully installed pandas>=2.0
Installing matplotlib>=3.7...
✅ Successfully installed matplotlib>=3.7
Installing pyyaml>=6.0...
✅ Successfully installed pyyaml>=6.0
Installing numba>=0.59...
✅ Successfully installed numba>=0.59
Installing h5py>=3.7...
✅ Successfully installed h5py>=3.7
Installing tqdm>=4.65...
✅ Successfully installed tqdm>=4.65
Installing psutil>=5.9...
✅ Successfully installed psutil>=5.9
Installing scienceplots>=2.0...
✅ Successfully installed scienceplots>=2.0
Installing qutip>=5.2...
✅ Successfully installed qutip>=5.2
✅ Setup complete.
📁 Working directory: /content/Quantum_Agrivoltaic_PT-HOPS/Redac_Paper1/quantum_simulations_framework


In [28]:
# 📦 Manual Source Installation for MesoHOPS (MesoscienceLab)
import sys, os, subprocess, shutil, urllib.request

def manual_install_mesohops():
    if "google.colab" not in sys.modules:
        print("Local environment assumed.")
        return

    # Try both common branch names for the archive under the new organization URL
    urls = [
        "https://github.com/MesoscienceLab/mesohops/archive/refs/heads/main.tar.gz",
        "https://github.com/MesoscienceLab/mesohops/archive/refs/heads/master.tar.gz"
    ]
    target_tar = "/content/mesohops.tar.gz"
    extract_dir = "/content/mesohops_source"

    success = False
    for url in urls:
        print(f"Attempting direct download: {url}")
        try:
            opener = urllib.request.build_opener()
            opener.addheaders = [('User-agent', 'Mozilla/5.0')]
            urllib.request.install_opener(opener)
            urllib.request.urlretrieve(url, target_tar)
            print(f"✅ Download successful from {url}")
            success = True
            break
        except Exception as e:
            print(f"⚠️  Download from {url} failed: {e}")

    if not success:
        raise RuntimeError("Could not download MesoHOPS from MesoscienceLab. Please check the branch name or URL.")

    try:
        # Step 2: Extract
        if os.path.exists(extract_dir): shutil.rmtree(extract_dir)
        os.makedirs(extract_dir)
        import tarfile
        with tarfile.open(target_tar, "r:gz") as tar:
            tar.extractall(path=extract_dir)

        # Find the actual source folder inside the extracted path
        inner_dir = next(os.path.join(extract_dir, d) for d in os.listdir(extract_dir) if os.path.isdir(os.path.join(extract_dir, d)))
        print(f"✅ Extraction successful to {inner_dir}")

        # Step 3: Local installation
        print("Installing package from local source...")
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", inner_dir], check=True)
        print("✅ MesoHOPS installed successfully.")

    except Exception as e:
        print(f"❌ Manual installation failed: {e}")
        raise

manual_install_mesohops()

Attempting direct download: https://github.com/MesoscienceLab/mesohops/archive/refs/heads/main.tar.gz
⚠️  Download from https://github.com/MesoscienceLab/mesohops/archive/refs/heads/main.tar.gz failed: HTTP Error 404: Not Found
Attempting direct download: https://github.com/MesoscienceLab/mesohops/archive/refs/heads/master.tar.gz
✅ Download successful from https://github.com/MesoscienceLab/mesohops/archive/refs/heads/master.tar.gz
✅ Extraction successful to /content/mesohops_source/mesohops-master
Installing package from local source...


/tmp/ipykernel_784/2843713559.py:40: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)


✅ MesoHOPS installed successfully.


In [29]:
# ── Cell 2: Load & validate parameters.yaml ──────────────────────────────────
import yaml

with open('parameters.yaml') as f:
    config = yaml.safe_load(f)

print(f"Project : {config['metadata']['project']}")
print(f"L       : {config['dynamics']['hierarchy_depth']}")
print(f"K       : {config['dynamics']['matsubara_truncation']}")
print(f"Δt      : {config['dynamics']['time_step']} fs")
print(f"T       : {config['bath']['temperature']} K")
print(f"n_traj  : {config.get('simulation', {}).get('n_traj', 100)}")

Project : Quantum_Agrivoltaic_PT-HOPS
L       : 10
K       : 10
Δt      : 0.5 fs
T       : 295.0 K
n_traj  : 100


In [30]:
# ── Cell 3: Override n_traj for Colab (reduce to avoid OOM) ─────────────────
# Change N_TRAJ to 100 for publication-grade results (requires ~10 GB RAM/traj)
N_TRAJ = 5  # safe for Colab (~12 GB RAM); use 100 on local hardware

config.setdefault('simulation', {})['n_traj'] = N_TRAJ
print(f'n_traj set to {N_TRAJ} for this run.')

n_traj set to 5 for this run.


In [31]:
# ── Cell 4: Check MesoHOPS availability ──────────────────────────────────────
from reproducibility.main import check_environment
assert check_environment(), '❌ MesoHOPS not available — re-run Cell 1'

AttributeError: 'OutStream' object has no attribute 'reconfigure'

In [34]:
import os

# Patch reproducibility/main.py to remove the unsupported .reconfigure() calls
filepath = 'reproducibility/main.py'
if os.path.exists(filepath):
    with open(filepath, 'r') as f:
        lines = f.readlines()

    with open(filepath, 'w') as f:
        for line in lines:
            if 'sys.stdout.reconfigure' in line or 'sys.stderr.reconfigure' in line:
                f.write(f'# {line}')  # Comment out the problematic lines
            else:
                f.write(line)
    print(f'✅ Patched {filepath} to fix AttributeError.')
else:
    print(f'❌ Could not find {filepath}')

✅ Patched reproducibility/main.py to fix AttributeError.


In [35]:
# ── Cell 5: Convergence audit (L=9, 10, 11) ──────────────────────────────────
# Uncomment to run; takes ~3 min on Colab
# from reproducibility.main import run_convergence_audit
# audit_data = run_convergence_audit()
# print(f"MAE(L10→L11): {audit_data['audit_mae_10_11']:.2e}")
print('ℹ️  Convergence audit skipped (uncomment to run).')

ℹ️  Convergence audit skipped (uncomment to run).


In [ ]:
# ── Cell 6: Full FMO simulation (filtered + broadband ensemble) ──────────────
from reproducibility.main import run_full_fmo_simulation
sim_results, time_points = run_full_fmo_simulation(config)
print('✅ Simulation complete.')


[Step 3] Running full FMO simulation (filtered + broadband, ensemble)...
  Running filtered excitation (5 trajectories)...


filtered:   0%|          | 0/5 [00:00<?, ?traj/s]

Integration from  0  to  1000.0
Noise Model initialized with SEED =  42
* Negative values will be set to 0.
max negative: -6615.629215546103
fractional negative: 2.4846875444313463e-05
negative number: 13993


In [ ]:
# ── Cell 7: Generate & display figures ───────────────────────────────────────
from reproducibility.main import generate_figures
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

generate_figures(config, sim_results, time_points)

jpcl_dir = os.path.abspath('../../Theory_Journals/JPCL')
for pattern, title in [
    ('Quantum_dynamics_*.png', 'Figure 1 — Quantum Dynamics'),
    ('ETR_Under_Environmental_Effects*.png', 'Figure 2 — Environmental Robustness'),
]:
    files = sorted(glob.glob(os.path.join(jpcl_dir, pattern)))
    if files:
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.imshow(mpimg.imread(files[-1]))
        ax.axis('off')
        ax.set_title(title)
        plt.tight_layout()
        plt.show()
    else:
        print(f'⚠️  {title} not found in {jpcl_dir}')